<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/MC12_ReAct_RAG_HR_Agent_Colab_2026_06_27_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MC12 — ReAct Agent + RAG over HR Data (Colab) — v2
### Decoding Data Science · AI Residency Masterclass 12

> 🔁 **If you already ran the older version in this session and hit a numpy/pandas error:**
> go to **Runtime → Restart session**, then run this notebook from the top. Do **not** run any
> `pip install --force-reinstall numpy pandas` — that breaks Colab's environment.

We build the exact pattern from the slides: a **ReAct agent** with two tools that decides which to use.

1. **`hr_policy`** — a **LlamaIndex RAG** query engine over a sample leave-**policy** document.
2. **`get_leave_balance`** — a Python function that looks up an employee's **leave balance**.

You'll watch the **Thought → Action → Observation** loop stream out live. That loop *is* the agent.

> ⚠️ Policy text and balances below are **fictional sample data** for teaching — not real HR policy.

## Step 0 — Install (current LlamaIndex, no version pins)
We install the latest LlamaIndex — it works with Colab's numpy 2 / pandas 3. **Takes ~40–60s.**

In [1]:
!pip install -q llama-index
print("\n✅ Installed. No restart needed (unless you ran a broken install earlier — then Runtime → Restart session).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.6/173.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 10.7 MB/s eta 0:00:00

✅ Installed. No restart needed (unless you ran a broken install earlier — then Runtime → Restart session).


## Step 1 — Your OpenAI key (from Colab Secrets)
Keep your key **out of the code**:

1. Click the 🔑 **key icon** in the left sidebar.
2. **+ Add new secret** → name it exactly `OPENAI_API_KEY`, paste your key.
3. Toggle **Notebook access ON**.

In [2]:
from google.colab import userdata
import os, warnings
warnings.filterwarnings("ignore")  # keep the teaching output clean

os.environ["OPENAI_API_KEY"] = userdata.get("openai")
print("✅ Key loaded from Colab Secrets." if os.environ.get("OPENAI_API_KEY") else "❌ No key found — re-check Step 1.")

✅ Key loaded from Colab Secrets.


## Step 2 — Sample data: the policy + the balances
Two kinds of knowledge the model can't know on its own — so it will have to *act* to get them.
(No pandas needed — we keep it plain to avoid any environment issues.)

In [3]:
from llama_index.core import Document

# --- (A) HR LEAVE POLICY  (unstructured text -> goes into RAG) ---
policy_docs = [
    Document(text=("ANNUAL LEAVE. Full-time employees are entitled to 22 working days of paid annual "
                   "leave per year, accrued monthly. Requests must be submitted at least 7 days in "
                   "advance through the HR portal and require line-manager approval.")),
    Document(text=("SICK LEAVE. Employees receive up to 15 days of fully paid sick leave per year. "
                   "A medical certificate is required for any absence longer than 2 consecutive days.")),
    Document(text=("PARENTAL LEAVE. Maternity leave is 60 calendar days at full pay. Paternity leave is "
                   "5 working days at full pay. Employees must notify HR at least 30 days before the "
                   "expected date of leave.")),
    Document(text=("CARRY-OVER. A maximum of 10 unused annual-leave days may be carried into the next "
                   "year. Any carried-over days unused by the end of Q1 (31 March) expire.")),
    Document(text=("PUBLIC HOLIDAYS. Official public holidays follow the UAE national calendar and are "
                   "NOT deducted from an employee's annual-leave balance.")),
    Document(text=("BEREAVEMENT LEAVE. Employees are granted 5 working days of paid leave in the event "
                   "of an immediate family member's death. Documentation may be required.")),
]

# --- (B) LEAVE BALANCES  (structured data -> goes into a function tool) ---
leave_balances = {
    1024: {"name": "Aisha Khan",   "annual_remaining": 12, "annual_total": 22, "sick_remaining": 15},
    1025: {"name": "John Doe",     "annual_remaining":  3, "annual_total": 22, "sick_remaining": 10},
    1026: {"name": "Maria Garcia", "annual_remaining": 18, "annual_total": 22, "sick_remaining": 15},
    1027: {"name": "David Lee",    "annual_remaining": 20, "annual_total": 22, "sick_remaining": 14},
}

print(f"Policy chunks: {len(policy_docs)}\n")
print(f"{'ID':<6}{'Name':<16}{'Annual left':<14}{'Sick left'}")
for eid, r in leave_balances.items():
    print(f"'{eid:<6}{r['name']:<16}{str(r['annual_remaining'])+'/'+str(r['annual_total']):<14}{r['sick_remaining']}")

Policy chunks: 6

ID    Name            Annual left   Sick left
'1024  Aisha Khan      12/22         15
'1025  John Doe        3/22          10
'1026  Maria Garcia    18/22         15
'1027  David Lee       20/22         14


## Step 3 — Point LlamaIndex at OpenAI
`Settings` is the global config: which LLM reasons, which model embeds.

In [4]:
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)              # the "brain" that reasons (ReAct)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small") # turns docs into vectors

print("✅ LLM:", Settings.llm.model, "| Embeddings: text-embedding-3-small")

✅ LLM: gpt-4o-mini | Embeddings: text-embedding-3-small


## Step 4 — Build the RAG index over the policy (LlamaIndex)
Ingest → embed → index → query engine. This is the **R**etrieval in RAG.

In [5]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(policy_docs)        # ingest + embed + index
policy_query_engine = index.as_query_engine(similarity_top_k=2)

# sanity check (plain RAG, no agent yet)
print(policy_query_engine.query("How many days of paternity leave do we get?"))

You get 5 working days of paternity leave at full pay.


In [7]:
leave_balances

{1024: {'name': 'Aisha Khan',
  'annual_remaining': 12,
  'annual_total': 22,
  'sick_remaining': 15},
 1025: {'name': 'John Doe',
  'annual_remaining': 3,
  'annual_total': 22,
  'sick_remaining': 10},
 1026: {'name': 'Maria Garcia',
  'annual_remaining': 18,
  'annual_total': 22,
  'sick_remaining': 15},
 1027: {'name': 'David Lee',
  'annual_remaining': 20,
  'annual_total': 22,
  'sick_remaining': 14}}

## Step 5 — Turn the query engine into a TOOL 🔗
The key idea from the talk: **wrap the LlamaIndex query engine as a tool** the agent can call.
The `description` is what the agent reads to decide *when* to use it — write it carefully.

In [6]:
from llama_index.core.tools import QueryEngineTool

policy_tool = QueryEngineTool.from_defaults(
    query_engine=policy_query_engine,
    name="hr_policy",
    description=("Answers questions about company HR LEAVE POLICY rules: annual leave, sick leave, "
                 "parental/maternity/paternity leave, carry-over, and public holidays. "
                 "Input: a plain-text question."),
)
print("✅ RAG query engine is now a tool:", policy_tool.metadata.name)

✅ RAG query engine is now a tool: hr_policy


## Step 6 — Build the leave-balance TOOL (a plain function)
Any Python function can become a tool — LlamaIndex reads the **type hints** and **docstring**
to describe it to the model.

In [8]:
from llama_index.core.tools import FunctionTool

def get_leave_balance(employee_id: int) -> str:
    """Look up the remaining LEAVE BALANCE (days left) for a specific employee by their employee_id."""
    rec = leave_balances.get(int(employee_id))
    if not rec:
        return f"No employee found with id {employee_id}."
    return (f"{rec['name']} (ID {employee_id}): {rec['annual_remaining']} of {rec['annual_total']} "
            f"annual-leave days remaining; {rec['sick_remaining']} sick-leave days remaining.")

balance_tool = FunctionTool.from_defaults(fn=get_leave_balance)
print("✅ Function is now a tool:", balance_tool.metadata.name)

✅ Function is now a tool: get_leave_balance


## Step 7 — Assemble the ReAct agent 🧠
We use the current LlamaIndex **workflow** `ReActAgent` (async). The little `ask()` helper below
**streams the Thought / Action / Observation loop** so you can watch it reason in real time.

In [9]:
from llama_index.core.agent.workflow import ReActAgent, AgentStream, ToolCallResult

agent = ReActAgent(
    tools=[policy_tool, balance_tool],
    llm=Settings.llm,
    system_prompt=("You are an HR assistant. Use the tools to answer questions about leave policy "
                   "and individual employee leave balances. Use a tool whenever the question is about "
                   "policy details or a specific employee's balance."),
)

async def ask(question: str):
    """Run the agent and stream its reasoning (Thought/Action) + tool results (Observation)."""
    print(f"❓ {question}\n" + "-" * 72)
    handler = agent.run(question)
    async for ev in handler.stream_events():
        if isinstance(ev, AgentStream):
            print(ev.delta, end="", flush=True)            # Thought / Action / Action Input / Answer
        elif isinstance(ev, ToolCallResult):
            print(f"\n   🔧 OBSERVATION [{ev.tool_name}] -> {ev.tool_output}\n")
    final = await handler
    print("\n\n🟢 FINAL ANSWER:\n", final)
    return final

print("✅ Agent ready with tools: hr_policy, get_leave_balance")

✅ Agent ready with tools: hr_policy, get_leave_balance


## Step 8 — A POLICY question (uses RAG only)
The agent reasons it needs `hr_policy`, calls it, observes the retrieved answer, then responds.

In [10]:
await ask("What is our parental leave policy?")

❓ What is our parental leave policy?
------------------------------------------------------------------------
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: hr_policy
Action Input: {"input":"What is our parental leave policy?"}
   🔧 OBSERVATION [hr_policy] -> The parental leave policy includes maternity leave of 60 calendar days at full pay and paternity leave of 5 working days at full pay. Employees are required to notify HR at least 30 days before the expected date of leave.

Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: Our parental leave policy includes maternity leave of 60 calendar days at full pay and paternity leave of 5 working days at full pay. Employees are required to notify HR at least 30 days before the expected date of leave.

🟢 FINAL ANSWER:
 Our parental leave policy includes maternity leave of 60 calendar days at full pay and paternity leave of 5 

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Our parental leave policy includes maternity leave of 60 calendar days at full pay and paternity leave of 5 working days at full pay. Employees are required to notify HR at least 30 days before the expected date of leave.')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-EUrqO8qyz0f8XwhOzN3hy1UkpC8E5', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'created': 1791025524, 'model': 'gpt-4o-mini-2024-07-18', 'object': 'chat.completion.chunk', 'moderation': None, 'service_tier': 'default', 'system_fingerprint': 'fp_d0d66bff01', 'usage': None, 'obfuscation': 'YE'}, tool_calls=[ToolCallResult(tool_name='hr_policy', tool_kwargs={'input': 'What is our parental leave policy?'}, tool_id='bb230af0-7732-4c4

## Step 9 — A DATA question (uses the function tool only)
The model can't know this from training — so it **acts** by calling `get_leave_balance`.

In [11]:
await ask("How many annual leave days does employee 1024 have left?")

❓ How many annual leave days does employee 1024 have left?
------------------------------------------------------------------------
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: get_leave_balance
Action Input: {"employee_id": 1024}
   🔧 OBSERVATION [get_leave_balance] -> Aisha Khan (ID 1024): 12 of 22 annual-leave days remaining; 15 sick-leave days remaining.

Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: Employee 1024, Aisha Khan, has 12 out of 22 annual leave days remaining.

🟢 FINAL ANSWER:
 Employee 1024, Aisha Khan, has 12 out of 22 annual leave days remaining.


AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Employee 1024, Aisha Khan, has 12 out of 22 annual leave days remaining.')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-EUrr8tKNY6Mvk68WvLC971sgO9vrQ', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'created': 1791025570, 'model': 'gpt-4o-mini-2024-07-18', 'object': 'chat.completion.chunk', 'moderation': None, 'service_tier': 'default', 'system_fingerprint': 'fp_d0d66bff01', 'usage': None, 'obfuscation': '8R'}, tool_calls=[ToolCallResult(tool_name='get_leave_balance', tool_kwargs={'employee_id': 1024}, tool_id='011a5b4e-f3ec-4a0b-b929-13d99fc2b803', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='Aisha Khan (ID 1024): 12 of 22 annual-leave days remaining; 15 sick-leave days 

## Step 10 — The HR Chatbot question (BOTH tools) 🌟
The exact example from the slides. One question needs **two tools** — watch the loop run twice:
retrieve the policy *and* look up the balance, then combine them.

In [ ]:
await ask("What's our parental leave policy, and how many annual leave days does employee 1024 have left?")

❓ What's our parental leave policy, and how many annual leave days does employee 1024 have left?
------------------------------------------------------------------------
Thought: The current language of the user is: English. I need to use a tool to help me answer the question about the parental leave policy.
Action: hr_policy
Action Input: {"input":"What is the parental leave policy?"}
   🔧 OBSERVATION [hr_policy] -> Maternity leave is 60 calendar days at full pay, while paternity leave is 5 working days at full pay. Employees are required to notify HR at least 30 days before the expected date of leave.

Thought: I have the information about the parental leave policy. Now, I will check the annual leave balance for employee 1024.
Action: get_leave_balance
Action Input: {'employee_id': 1024}
   🔧 OBSERVATION [get_leave_balance] -> Aisha Khan (ID 1024): 12 of 22 annual-leave days remaining; 15 sick-leave days remaining.

Thought: I can answer without using any more tools. I'll use the use

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Our parental leave policy states that maternity leave is 60 calendar days at full pay, while paternity leave is 5 working days at full pay. Employees must notify HR at least 30 days before the expected date of leave. As for employee 1024, Aisha Khan, she has 12 out of 22 annual leave days remaining.')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-DvOJL6lGGtQd7tAYLMfaPI0Xcc0XY', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'created': 1782570519, 'model': 'gpt-4o-mini-2024-07-18', 'object': 'chat.completion.chunk', 'moderation': None, 'service_tier': 'default', 'system_fingerprint': 'fp_86eb24cbce', 'usage': None, 'obfuscation': 'xF'}, tool_calls=[ToolCallResult(tool_name='hr_policy', tool_kwa

## Step 11 — Try your own questions
Type a question (e.g. *"Can I carry over unused leave?"*, *"Balance for employee 1025?"*). Type `quit` to stop.

In [12]:
while True:
    q = input("\nAsk the HR agent (or 'quit'): ").strip()
    if q.lower() in {"quit", "exit", ""}:
        print("Done.")
        break
    await ask(q)


Ask the HR agent (or 'quit'): how many leaves does 1025 have and how many he can take this year
❓ how many leaves does 1025 have and how many he can take this year
------------------------------------------------------------------------
Thought: The user is asking for the leave balance of employee 1025 and how many leaves they can take this year. I need to look up the leave balance for this specific employee.
Action: get_leave_balance
Action Input: {"employee_id": 1025}
   🔧 OBSERVATION [get_leave_balance] -> John Doe (ID 1025): 3 of 22 annual-leave days remaining; 10 sick-leave days remaining.

Thought: I have the leave balance for employee 1025. They have 3 of 22 annual leave days remaining and 10 sick leave days remaining. Now, I need to determine how many leaves they can take this year.
Answer: Employee 1025, John Doe, has 3 annual leave days remaining out of 22 for the year, and he can also take 10 sick leave days.

🟢 FINAL ANSWER:
 Employee 1025, John Doe, has 3 annual leave day

---
### What you just saw, mapped to the deck
| In the stream | In the talk |
|---|---|
| `Thought:` | **Chain of Thought** — the model reasoning about what to do |
| `Action:` / `Action Input:` | **ReAct** — it *acts* by calling a tool |
| `🔧 OBSERVATION` | the result it reads back, grounding the next thought |
| `hr_policy` tool | **LlamaIndex RAG** over your documents |
| `get_leave_balance` tool | a live-data lookup (stand-in for your HR system / API) |

**To make it real:** swap `policy_docs` for `SimpleDirectoryReader("your_folder").load_data()` to load your
own PDFs, and replace `leave_balances` with a real database/API call inside `get_leave_balance`.

*The same pattern in LangChain: `create_react_agent(llm, tools)` — same loop, different framework.*

— Built for **AI Residency Cohort 11** · Decoding Data Science